In [2]:
import cv2
import numpy as np
import torch
import os
import imageio
# from run_utils.data_utils import write_video_imageio

def write_video_imageio(video, output_path, fps=24):
    os.makedirs(os.path.dirname(output_path), exist_ok=True)
    imageio.mimsave(output_path, video.astype(np.uint8), fps=fps)

def load_video_simple(video_path):
        # load video
    cap = cv2.VideoCapture(video_path)
    fps = cap.get(cv2.CAP_PROP_FPS)
    frames = []
    while 1:
        ret, frame = cap.read()
        if not ret:
            break
        frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        frames.append(frame)
    cap.release()

    video_tensor = torch.tensor(np.array(frames)).permute(0, 3, 1, 2).float() / 255.0
    video_tensor = video_tensor.unsqueeze(0).permute(0, 2, 1, 3, 4)

    print(f"Min: {video_tensor.min()}, Max: {video_tensor.max()}")

    return video_tensor, np.array(frames), fps

def load_video(
    path,
    max_h=None,
    fps_jump=None,
    max_frames=None,
    start_frame=0,
    end_frame=None,
    res_h=None,
    res_w=None,
    vae_spatial_reduction=8,
    vae_temporal_reduction=4,
    force_h=False,
    default_fps=30.0
):
    """
    Load a video file or a directory of frames and return a tensor of the frames.
    Args:
        path (str): Path to the video file, or a directory of frames.
        max_h (int): Maximum height of the frames. If None, no resizing is done.
        fps_jump (int): Number of frames to skip. If None, no skipping is done.
        max_frames (int): Maximum number of frames to load. If None, all frames are loaded.
        vae_spatial_reduction (int): Spatial reduction factor for VAE.
        vae_temporal_reduction (int): Temporal reduction factor for VAE.
        force_h (bool): If True, force the height to be max_h even if it is smaller than the original height.
        default_fps (float): Default FPS to use if loading from frames.
    Returns:
        video_tensor (torch.Tensor): Tensor of the frames.
        frames (list): List of the frames. type: np.ndarray, shape: (T, H, W, C), dtype: uint8, values in [0, 255]
        vid_data (dict): Dictionary containing video metadata (fps, new_fps).
    """
    frames = []
    vid_data = {}

    if os.path.isdir(path):
        # Directory of frames
        frame_files = sorted([
            os.path.join(path, f)
            for f in os.listdir(path)
            if f.lower().endswith(('.png', '.jpg', '.jpeg', '.bmp'))
        ])
        for frame_file in frame_files:
            frame = cv2.imread(frame_file)
            frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            h, w = frame.shape[:2]
            if res_h is not None and res_w is not None:
                new_h = res_h
                new_w = res_w
            elif (max_h is not None) and (h > max_h or force_h):
                new_h = max_h
                new_w = round(w * max_h / h)
            else:
                new_h = h
                new_w = w
            new_h = new_h - new_h % vae_spatial_reduction
            new_w = new_w + (-new_w) % vae_spatial_reduction
            frame = cv2.resize(frame, (new_w, new_h))
            frames.append(frame)
        vid_data["fps"] = default_fps
    else:
        # Video file
        cap = cv2.VideoCapture(path)
        fps = cap.get(cv2.CAP_PROP_FPS)
        vid_data["fps"] = fps
        while True:
            ret, frame = cap.read()
            if not ret:
                break
            frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            h, w = frame.shape[:2]
            if res_h is not None and res_w is not None:
                new_h = res_h
                new_w = res_w
            elif (max_h is not None) and (h > max_h or force_h):
                new_h = max_h
                new_w = round(w * max_h / h)
            else:
                new_h = h
                new_w = w
            new_h = new_h - new_h % vae_spatial_reduction
            new_w = new_w + (-new_w) % vae_spatial_reduction
            frame = cv2.resize(frame, (new_w, new_h))
            frames.append(frame)
        cap.release()

    if fps_jump is not None:
        frames = frames[::fps_jump]
        vid_data["new_fps"] = vid_data["fps"] / fps_jump
    else:
        vid_data["new_fps"] = vid_data["fps"]

    if (end_frame is not None):
        frames = frames[:end_frame]
    if (start_frame is not None):
        frames = frames[start_frame:]
    if max_frames is not None:
        frames = frames[:max_frames]

    if len(frames) == 0:
        raise ValueError("No frames loaded from the given path.")

    frames = frames[:len(frames) - (len(frames) - 1) % vae_temporal_reduction]
    video_tensor = torch.tensor(np.array(frames)).permute(0, 3, 1, 2).float() / 255.0
    video_tensor = video_tensor.unsqueeze(0).permute(0, 2, 1, 3, 4)

    print(f"Min: {video_tensor.min()}, Max: {video_tensor.max()}")

    _, c, f, h, w = video_tensor.shape

    assert h % vae_spatial_reduction == 0, f"Height {h} is not divisible by vae_spatial_reduction {vae_spatial_reduction}"
    assert w % vae_spatial_reduction == 0, f"Width {w} is not divisible by vae_spatial_reduction {vae_spatial_reduction}"
    assert (f - 1) % vae_temporal_reduction == 0, f"Frames {f} is not divisible by vae_temporal_reduction {vae_temporal_reduction}"

    return video_tensor, frames, vid_data


In [2]:
vid_t, vid_np, fps = load_video_simple('/home/assafsin/projects/vdm_optical_flow/flow_samples/car_turn_video.mp4')
print(f"Video shape: {vid_t.shape}, numpy shape: {vid_np.shape}, FPS: {fps}")
write_video_imageio(vid_np, '/home/assafsin/projects/vdm_optical_flow/self_warp/car_turn_video.mp4', fps=fps)

RuntimeError: number of dims don't match in permute

In [13]:
vid_t, vid_list, fps = load_video('/home/assafsin/projects/motion_guidance/DAVIS/JPEGImages/480p/car-turn/',
                                res_h=480, res_w=720, start_frame=30, max_frames=49)
write_video_imageio(np.array(vid_list), '/home/assafsin/projects/vdm_optical_flow/self_warp/car_turn_video_sf=30_f=49.mp4', fps=8)

Min: 0.0, Max: 1.0


In [14]:
vid_t.shape, np.array(vid_list).shape, fps

(torch.Size([1, 3, 49, 480, 720]),
 (49, 480, 720, 3),
 {'fps': 30.0, 'new_fps': 30.0})

In [ ]:
from diffusers.utils import export_to_video, load_image, load_video
import PIL.Image

In [3]:
vid = load_video('/home/assafsin/projects/vdm_optical_flow/data/car_turn_video_sf=30_f=49.mp4')

In [5]:
type(vid), len(vid), vid[0].size

(list, 49, (720, 480))

In [ ]:
# save the first frame as an image
vid[0].save('/home/assafsin/projects/vdm_optical_flow/data/car_turn_video_frame_0.png')

In [11]:
vid_t, vid_list, fps = load_video('/home/assafsin/projects/motion_guidance/DAVIS/JPEGImages/480p/cows/',
                                res_h=480, res_w=720, start_frame=30, max_frames=49)
# write the video to a file
write_video_imageio(np.array(vid_list), '/home/assafsin/projects/vdm_optical_flow/data/cows_video_sf=30_f=49.mp4', fps=8)
# save the first frame as an image
from PIL import Image
Image.fromarray(vid_list[0]).save('/home/assafsin/projects/vdm_optical_flow/data/cows_video_frame_0.png')

Min: 0.0, Max: 1.0


In [ ]:
start_frame , max_frames = 5, 49
vid_name = 'train'
vid_t, vid_list, fps = load_video(f'/home/assafsin/projects/motion_guidance/DAVIS/JPEGImages/480p/{vid_name}/',
                                res_h=480, res_w=720, start_frame=start_frame, max_frames=max_frames)
# write the video to a file
write_video_imageio(np.array(vid_list), f'/home/assafsin/projects/vdm_optical_flow/data/{vid_name}_video_sf={start_frame}_f={max_frames}.mp4', fps=8)
# save the first frame as an image
from PIL import Image
Image.fromarray(vid_list[0]).save(f'/home/assafsin/projects/vdm_optical_flow/data/{vid_name}_video_frame_0.png')

Min: 0.0, Max: 1.0


In [3]:
start_frame , max_frames = 0, 49
vid_name = 'rhino'
vid_t, vid_list, fps = load_video(f'/home/assafsin/projects/motion_guidance/DAVIS/JPEGImages/480p/{vid_name}/',
                                res_h=480, res_w=720, start_frame=start_frame, max_frames=max_frames)
# write the video to a file
write_video_imageio(np.array(vid_list), f'/home/assafsin/projects/vdm_optical_flow/data/{vid_name}_video_sf={start_frame}_f={max_frames}.mp4', fps=8)
# save the first frame as an image
from PIL import Image
Image.fromarray(vid_list[0]).save(f'/home/assafsin/projects/vdm_optical_flow/data/{vid_name}_video_frame_0.png')

Min: 0.0, Max: 1.0


In [4]:
start_frame , max_frames = 0, 49
vid_name = 'locomotive'
vid_t, vid_list, fps = load_video('/home/assafsin/projects/vdm_optical_flow/data/locomotive.mp4',
                                res_h=480, res_w=720, start_frame=start_frame, max_frames=max_frames)
# write the video to a file
write_video_imageio(np.array(vid_list), f'/home/assafsin/projects/vdm_optical_flow/data/{vid_name}_video_sf={start_frame}_f={max_frames}.mp4', fps=8)
# save the first frame as an image
from PIL import Image
Image.fromarray(vid_list[0]).save(f'/home/assafsin/projects/vdm_optical_flow/data/{vid_name}_video_frame_0.png')

Min: 0.0, Max: 1.0


In [3]:
# iterate over files in /home/assafsin/projects/vdm_optical_flow/data/ and duplicate and rename all file that contains "girl_mount_v3" to "girl_mount_v3_p2"importlib
import shutil
import os

data_dir = "/home/assafsin/projects/vdm_optical_flow/data/"
for filename in os.listdir(data_dir):
    if "girl_mount_v3" in filename:
        new_filename = filename.replace("girl_mount_v3", "girl_mount_v3_p2")
        print(f"Copying {filename} to {new_filename}")
        shutil.copy(os.path.join(data_dir, filename), os.path.join(data_dir, new_filename))


Copying girl_mount_v3_backward_direct_flows.npy to girl_mount_v3_p2_backward_direct_flows.npy


Copying girl_mount_v3_backward_direct_occlusion.npy to girl_mount_v3_p2_backward_direct_occlusion.npy
Copying girl_mount_v3_consecutive_flows.npy to girl_mount_v3_p2_consecutive_flows.npy
Copying girl_mount_v3_direct_flows.npy to girl_mount_v3_p2_direct_flows.npy
Copying girl_mount_v3_direct_occlusion.npy to girl_mount_v3_p2_direct_occlusion.npy
Copying girl_mount_v3_sf=0_f=49.mp4 to girl_mount_v3_p2_sf=0_f=49.mp4
Copying girl_mount_v3_video_frame_0.png to girl_mount_v3_p2_video_frame_0.png


In [6]:
# iterate over files in /home/assafsin/projects/vdm_optical_flow/data/ and duplicate and rename all files that contain "girl_mount" but not "girl_mount_v3", to "girl_mount_p2"
import shutil
import os

data_dir = "/home/assafsin/projects/vdm_optical_flow/data/"
for filename in os.listdir(data_dir):
    if "girl_mount" in filename and "girl_mount_v3" not in filename:
        new_filename = filename.replace("girl_mount", "girl_mount_p2")
        print(f"Copying {filename} to {new_filename}")
        shutil.copy(os.path.join(data_dir, filename), os.path.join(data_dir, new_filename))

Copying girl_mount_backward_direct_flows.npy to girl_mount_p2_backward_direct_flows.npy


Copying girl_mount_backward_direct_occlusion.npy to girl_mount_p2_backward_direct_occlusion.npy
Copying girl_mount_consecutive_flows.npy to girl_mount_p2_consecutive_flows.npy
Copying girl_mount_direct_flows.npy to girl_mount_p2_direct_flows.npy
Copying girl_mount_direct_occlusion.npy to girl_mount_p2_direct_occlusion.npy
Copying girl_mount_sf=0_f=49.mp4 to girl_mount_p2_sf=0_f=49.mp4
Copying girl_mount_video_frame_0.png to girl_mount_p2_video_frame_0.png


In [1]:
import numpy as np
a = np.load("/home/assafsin/projects/vdm_optical_flow/consecutive_flows.npy")
a.shape

ValueError: cannot reshape array of size 19791840 into shape (48,480,720,2)

In [6]:
start_frame , max_frames = 0, 49
fps_jump = 2
vid_name = 'locomotive-960×540'
vid_t, vid_list, fps = load_video('/home/assafsin/projects/vdm_optical_flow/data/locomotive-960×540.mp4',
                                res_h=480, res_w=720, start_frame=start_frame, max_frames=max_frames, fps_jump=fps_jump)
# write the video to a file
write_video_imageio(np.array(vid_list), f'/home/assafsin/projects/vdm_optical_flow/data/{vid_name}_video_sf={start_frame}_f={max_frames}_fps_jump={fps_jump}.mp4', fps=8)
# save the first frame as an image
from PIL import Image
Image.fromarray(vid_list[0]).save(f'/home/assafsin/projects/vdm_optical_flow/data/{vid_name}_video_frame_0.png')

Min: 0.0, Max: 1.0


In [10]:
start_frame , max_frames = 0, 49
fps_jump = 4
vid_name = 'locomotive-960×540'
vid_t, vid_list, fps = load_video('/home/assafsin/projects/vdm_optical_flow/data/drafts/locomotive-960×540.mp4',
                                res_h=480, res_w=720, start_frame=start_frame, max_frames=max_frames, fps_jump=fps_jump)
# write the video to a file
write_video_imageio(np.array(vid_list), f'/home/assafsin/projects/vdm_optical_flow/data/{vid_name}_video_sf={start_frame}_f={max_frames}_fps_jump={fps_jump}.mp4', fps=8)
# save the first frame as an image
from PIL import Image
Image.fromarray(vid_list[0]).save(f'/home/assafsin/projects/vdm_optical_flow/data/{vid_name}_video_frame_0.png')

Min: 0.0, Max: 1.0


In [5]:
vid_t, vid_list, fps = load_video('/home/assafsin/projects/vdm_optical_flow/data/drafts/monkey_crop.mp4')

Min: 0.0, Max: 1.0


In [6]:
fps, vid_t.shape, np.array(vid_list).shape

({'fps': 30.104895104895103, 'new_fps': 30.104895104895103},
 torch.Size([1, 3, 285, 1248, 1080]),
 (285, 1248, 1080, 3))

In [9]:
start_frame , max_frames = 2, 49
fps_jump = 4
vid_name = 'monkey_crop1'
vid_t, vid_list, fps = load_video(f'/home/assafsin/projects/vdm_optical_flow/data/drafts/{vid_name}.mp4',
                                res_h=480, res_w=720, start_frame=start_frame, max_frames=max_frames, fps_jump=fps_jump)
# write the video to a file
write_video_imageio(np.array(vid_list), f'/home/assafsin/projects/vdm_optical_flow/data/{vid_name}_video_sf={start_frame}_f={max_frames}_fps_jump={fps_jump}.mp4', fps=8)
# save the first frame as an image
from PIL import Image
Image.fromarray(vid_list[0]).save(f'/home/assafsin/projects/vdm_optical_flow/data/{vid_name}_video_frame_0.png')

Min: 0.0, Max: 1.0


In [4]:
fps

{'fps': 30.104895104895103, 'new_fps': 15.052447552447552}

In [4]:
fps

{'fps': 30.0, 'new_fps': 30.0}

In [5]:
vid_t, vid_list, fps = load_video('/home/assafsin/projects/motion_guidance/DAVIS/JPEGImages/480p/gold-fish/')
write_video_imageio(np.array(vid_list), '/home/assafsin/projects/vdm_optical_flow/self_warp/gold-fish.mp4', fps=fps['fps'])

IMAGEIO FFMPEG_WRITER WARNING: input image is not divisible by macro_block_size=16, resizing from (856, 480) to (864, 480) to ensure video compatibility with most codecs and players. To prevent resizing, make your input image divisible by the macro_block_size or set the macro_block_size to 1 (risking incompatibility).


Min: 0.0, Max: 1.0


[swscaler @ 0x628b140] Warning: data is not aligned! This can lead to a speed loss


In [6]:
vid_t, vid_list, fps = load_video('/home/assafsin/projects/vdm_optical_flow/data/drafts/t2v_motion_transfer_train (online-video-cutter.com) (1).mp4')
fps, vid_t.shape, np.array(vid_list).shape

Min: 0.0, Max: 1.0


({'fps': 24.24742268041237, 'new_fps': 24.24742268041237},
 torch.Size([1, 3, 97, 480, 720]),
 (97, 480, 720, 3))

In [5]:
vid_t, vid_list, fps = load_video('/home/assafsin/projects/vdm_optical_flow/data/drafts/hhq_synthetics_018.mp4')
fps, vid_t.shape, np.array(vid_list).shape

Min: 0.0, Max: 1.0


({'fps': 16.0, 'new_fps': 16.0},
 torch.Size([1, 3, 49, 480, 3600]),
 (49, 480, 3600, 3))

In [5]:
vid_t, vid_list, fps = load_video('/home/assafsin/projects/vdm_optical_flow/data/drafts/cut_and_drag-duck.mp4')
fps, vid_t.shape, np.array(vid_list).shape

Min: 0.0, Max: 1.0


({'fps': 16.333333333333332, 'new_fps': 16.333333333333332},
 torch.Size([1, 3, 45, 480, 728]),
 (45, 480, 728, 3))

In [6]:
start_frame , max_frames = 0, 49
fps_jump = 1
vid_name = 'cut_and_drag-duck'
vid_t, vid_list, fps = load_video(f'/home/assafsin/projects/vdm_optical_flow/data/drafts/{vid_name}.mp4',
                                res_h=480, res_w=720, start_frame=start_frame, max_frames=max_frames, fps_jump=fps_jump)
# write the video to a file
write_video_imageio(np.array(vid_list), f'/home/assafsin/projects/vdm_optical_flow/data/{vid_name}_video_sf={start_frame}_f={max_frames}_fps_jump={fps_jump}.mp4', fps=8)
# save the first frame as an image
from PIL import Image
Image.fromarray(vid_list[0]).save(f'/home/assafsin/projects/vdm_optical_flow/data/{vid_name}_video_frame_0.png')

Min: 0.0, Max: 1.0


In [3]:
vid_t, vid_list, fps = load_video(f'/home/assafsin/projects/vdm_optical_flow/experiments/duck_i2v-flow_warp_guid/fw=60_wrp_w=7_mg=1000_gc=500_steps=200_r_steps=5_max_g_step=0.75_gt_flows_seed=42/motion_transfer.mp4')
# write the video to a file
write_video_imageio(np.array(vid_list), f'/home/assafsin/projects/vdm_optical_flow/experiments/duck_i2v-flow_warp_guid/fw=60_wrp_w=7_mg=1000_gc=500_steps=200_r_steps=5_max_g_step=0.75_gt_flows_seed=42/motion_transfer-fps=16.mp4', fps=16)

Min: 0.0, Max: 1.0


In [4]:
vid_t, vid_list, fps = load_video('/home/assafsin/projects/vdm_optical_flow/data/drafts/hhq_synthetics_025 (online-video-cutter.com).mp4')
print(fps, vid_t.shape, np.array(vid_list).shape)

from PIL import Image
Image.fromarray(vid_list[0]).save(f'/home/assafsin/projects/vdm_optical_flow/data/balloon_video_frame_0.png')

Min: 0.0, Max: 1.0
{'fps': 16.333333333333332, 'new_fps': 16.333333333333332} torch.Size([1, 3, 45, 480, 720]) (45, 480, 720, 3)


In [3]:
vid_t, vid_list, fps = load_video('/home/assafsin/projects/vdm_optical_flow/data/owl_sf=0_f=49.mp4')
print(fps, vid_t.shape, np.array(vid_list).shape)

from PIL import Image
Image.fromarray(vid_list[0]).save(f'/home/assafsin/projects/vdm_optical_flow/data/owl_video_frame_0.png')

Min: 0.0, Max: 1.0
{'fps': 60.0, 'new_fps': 60.0} torch.Size([1, 3, 49, 480, 720]) (49, 480, 720, 3)


In [4]:
vid_t, vid_list, fps = load_video('/home/assafsin/projects/vdm_optical_flow/data/girl_pearl_sf=0_f=49.mp4')
print(fps, vid_t.shape, np.array(vid_list).shape)

from PIL import Image
Image.fromarray(vid_list[0]).save(f'/home/assafsin/projects/vdm_optical_flow/data/girl_pearl_video_frame_0.png')

Min: 0.0, Max: 1.0
{'fps': 60.0, 'new_fps': 60.0} torch.Size([1, 3, 49, 480, 720]) (49, 480, 720, 3)


In [3]:
vid_t, vid_list, fps = load_video('/home/assafsin/projects/vdm_optical_flow/data/chess_sf=0_f=49.mp4')
print(fps, vid_t.shape, np.array(vid_list).shape)

from PIL import Image
Image.fromarray(vid_list[0]).save(f'/home/assafsin/projects/vdm_optical_flow/data/chess_video_frame_0.png')

Min: 0.0, Max: 1.0
{'fps': 60.0, 'new_fps': 60.0} torch.Size([1, 3, 49, 480, 720]) (49, 480, 720, 3)


In [5]:
vid_t, vid_list, fps = load_video('/home/assafsin/projects/vdm_optical_flow/data/drafts/hhq_synthetics_039 (online-video-cutter.com).mp4')
print(fps, vid_t.shape, np.array(vid_list).shape)

from PIL import Image
Image.fromarray(vid_list[0]).save(f'/home/assafsin/projects/vdm_optical_flow/data/hurricane_video_frame_0.png')

Min: 0.0, Max: 1.0
{'fps': 16.333333333333332, 'new_fps': 16.333333333333332} torch.Size([1, 3, 45, 480, 720]) (45, 480, 720, 3)


In [4]:
vid_t, vid_list, fps = load_video('/home/assafsin/projects/vdm_optical_flow/data/drafts/bear.mp4')
fps, vid_t.shape, np.array(vid_list).shape

Min: 0.0, Max: 1.0


({'fps': 16.0, 'new_fps': 16.0},
 torch.Size([1, 3, 77, 552, 1200]),
 (77, 552, 1200, 3))

In [1]:
import numpy as np
mask = np.load('/home/assaf.singer/projects/vdm_optical_flow/data/mountain_sf=10_f=49_fps_jump=3_mask.npy')

In [2]:
mask.shape

(49, 480, 720, 1)

In [ ]:
vid_t, vid_list, fps = load_video('/home/assafsin/projects/vdm_optical_flow/experiments-sdedit_blended_diffusion-new_mask/mountain_2_v2_i2v/mountain_2_v2_video_sf=10_f=49_fps_jump=3.mp4')
write_video_imageio(np.array(vid_list), f'/home/assafsin/projects/vdm_optical_flow/experiments-sdedit_blended_diffusion-new_mask/mountain_2_v2_i2v/mountain_2_v2_video_sf=10_f=49_fps_jump=3_fps=16.mp4', fps=16)

Min: 0.0, Max: 1.0


In [3]:
vid_t, vid_list, fps = load_video('/home/assafsin/projects/vdm_optical_flow/experiments-sdedit_blended_diffusion-new_mask/mountain_2_i2v/mountain_2_video_sf=0_f=49_fps_jump=2.mp4')
write_video_imageio(np.array(vid_list), f'/home/assafsin/projects/vdm_optical_flow/experiments-sdedit_blended_diffusion-new_mask/mountain_2_i2v/mountain_2_video_sf=0_f=49_fps_jump=2_fps=16.mp4', fps=16)

Min: 0.0, Max: 1.0


In [4]:
vid_t, vid_list, fps = load_video('/home/assafsin/projects/vdm_optical_flow/experiments-sdedit_blended_diffusion-new_mask/traffic_light_i2v/traffic_light_sf=0_f=49.mp4')
write_video_imageio(np.array(vid_list), f'/home/assafsin/projects/vdm_optical_flow/experiments-sdedit_blended_diffusion-new_mask/traffic_light_i2v/traffic_light_sf=0_f=49_fps=16.mp4', fps=16)

Min: 0.0, Max: 1.0


In [ ]:
vid_t, vid_list, fps = load_video(f'/home/assafsin/projects/vdm_optical_flow/experiments/duck_i2v-flow_warp_guid/fw=60_wrp_w=7_mg=1000_gc=500_steps=200_r_steps=5_max_g_step=0.75_gt_flows_seed=42/motion_transfer.mp4')
# write the video to a file
write_video_imageio(np.array(vid_list), f'/home/assafsin/projects/vdm_optical_flow/experiments/duck_i2v-flow_warp_guid/fw=60_wrp_w=7_mg=1000_gc=500_steps=200_r_steps=5_max_g_step=0.75_gt_flows_seed=42/motion_transfer-fps=16.mp4', fps=16)

Min: 0.0, Max: 1.0


In [7]:
vid_t, vid_list, fps = load_video(f'//home/assafsin/projects/vdm_optical_flow/experiments-sdedit_blended_diffusion-new_mask/chameleon_2_cos_i2v/chameleon_2_cos_sf=0_f=49.mp4')
# write the video to a file
write_video_imageio(np.array(vid_list), f'/home/assafsin/projects/vdm_optical_flow/experiments-sdedit_blended_diffusion-new_mask/chameleon_2_cos_i2v/chameleon_2_cos_sf=0_f=49_fps=16.mp4', fps=16)

Min: 0.0, Max: 1.0


In [8]:
vid_t, vid_list, fps = load_video(f'//home/assafsin/projects/vdm_optical_flow/experiments-sdedit_blended_diffusion-new_mask/chameleon_2_i2v/chameleon_2_sf=0_f=49.mp4')
# write the video to a file
write_video_imageio(np.array(vid_list), f'/home/assafsin/projects/vdm_optical_flow/experiments-sdedit_blended_diffusion-new_mask/chameleon_2_i2v/chameleon_2_sf=0_f=49_fps=16.mp4', fps=16)

Min: 0.0, Max: 1.0


In [9]:
vid_t, vid_list, fps = load_video(f'//home/assafsin/projects/vdm_optical_flow/experiments-sdedit_blended_diffusion-new_mask/chameleon_cos_i2v/chameleon_cos_sf=0_f=49.mp4')
# write the video to a file
write_video_imageio(np.array(vid_list), f'/home/assafsin/projects/vdm_optical_flow/experiments-sdedit_blended_diffusion-new_mask/chameleon_cos_i2v/chameleon_cos_sf=0_f=49_fps=16.mp4', fps=16)

Min: 0.0, Max: 1.0


In [10]:
vid_t, vid_list, fps = load_video(f'//home/assafsin/projects/vdm_optical_flow/experiments-sdedit_blended_diffusion-new_mask/chameleon_i2v/chameleon_sf=0_f=49.mp4')
# write the video to a file
write_video_imageio(np.array(vid_list), f'/home/assafsin/projects/vdm_optical_flow/experiments-sdedit_blended_diffusion-new_mask/chameleon_i2v/chameleon_sf=0_f=49_fps=16.mp4', fps=16)

Min: 0.0, Max: 1.0


In [3]:
vid_t, vid_list, fps = load_video(f'/home/assafsin/projects/vdm_optical_flow/data/sun_2_sf=0_f=49.mp4')
# write the video to a file
write_video_imageio(np.array(vid_list), f'/home/assafsin/projects/vdm_optical_flow/data/sun_2_sf=0_f=49_fps=16.mp4', fps=16)

Min: 0.0, Max: 1.0


In [11]:
vid_t, vid_list, fps = load_video('/home/assafsin/projects/vdm_optical_flow/data/chameleon_2_v2_cos_sf=0_f=49.mp4')
print(fps, vid_t.shape, np.array(vid_list).shape)

Min: 0.0, Max: 1.0
{'fps': 30.0, 'new_fps': 30.0} torch.Size([1, 3, 181, 480, 720]) (181, 480, 720, 3)


In [4]:
vid_t, vid_list, fps = load_video('/home/assafsin/projects/vdm_optical_flow/warp_video_framec-49-multipoints.mp4')
print(fps, vid_t.shape, np.array(vid_list).shape)

write_video_imageio(np.array(vid_list), '/home/assafsin/projects/vdm_optical_flow/warp_video_framec-49-multipoints-fixed.mp4')

Min: 0.0, Max: 1.0
{'fps': 8.0, 'new_fps': 8.0} torch.Size([1, 3, 49, 480, 720]) (49, 480, 720, 3)


In [5]:
vid_t, vid_list, fps = load_video('/home/assafsin/projects/vdm_optical_flow/mask_video_framec-49-multipoints.mp4')
print(fps, vid_t.shape, np.array(vid_list).shape)

write_video_imageio(np.array(vid_list), '/home/assafsin/projects/vdm_optical_flow/mask_video_framec-49-multipoints-fixed.mp4')

Min: 0.0, Max: 1.0
{'fps': 8.0, 'new_fps': 8.0} torch.Size([1, 3, 49, 480, 720]) (49, 480, 720, 3)


In [1]:
vid_t, vid_list, fps = load_video('/home/assafsin/projects/vdm_optical_flow/girljuggling_concatenated_fixed.mp4')
print(fps, vid_t.shape, np.array(vid_list).shape)

NameError: name 'load_video' is not defined

In [ ]:
# DL3DV Comparison video
vid_t, vid_list, fps = load_video('/home/assafsin/projects/vdm_optical_flow/concatenated_fixed.mp4')
print(fps, vid_t.shape, np.array(vid_list).shape)

Min: 0.0, Max: 1.0
{'fps': 16.0, 'new_fps': 16.0} torch.Size([1, 3, 49, 320, 1920]) (49, 320, 1920, 3)


In [4]:
vid_t, vid_list, fps = load_video('/home/assafsin/projects/vdm_optical_flow/for_paper-amir/RaceCars_our_i2v_output_sdedit7_blend14_seed0.mp4_concat.mp4')
print(fps, vid_t.shape, np.array(vid_list).shape)
write_video_imageio(np.array(vid_list), '/home/assafsin/projects/vdm_optical_flow/RaceCars_concat.mp4', fps=24)

Min: 0.0, Max: 1.0
{'fps': 16.0, 'new_fps': 16.0} torch.Size([1, 3, 81, 480, 1440]) (81, 480, 1440, 3)


In [5]:
vid_t, vid_list, fps = load_video('/home/assafsin/projects/vdm_optical_flow/for_paper-amir/RaceCars_our_i2v_output_sdedit7_blend14_seed0.mp4_concat.mp4')
print(fps, vid_t.shape, np.array(vid_list).shape)
write_video_imageio(np.array(vid_list), '/home/assafsin/projects/vdm_optical_flow/for_paper-amir/RaceCars_concat.mp4', fps=24)

Min: 0.0, Max: 1.0
{'fps': 16.0, 'new_fps': 16.0} torch.Size([1, 3, 81, 480, 1440]) (81, 480, 1440, 3)


In [6]:
vid_t, vid_list, fps = load_video('/home/assafsin/projects/vdm_optical_flow/for_paper-amir/Vines_our_i2v_output_sdedit6_blend12_seed0.mp4_concat.mp4')
print(fps, vid_t.shape, np.array(vid_list).shape)
write_video_imageio(np.array(vid_list), '/home/assafsin/projects/vdm_optical_flow/for_paper-amir/Vines_concat.mp4', fps=24)

Min: 0.0, Max: 1.0
{'fps': 16.0, 'new_fps': 16.0} torch.Size([1, 3, 81, 480, 1440]) (81, 480, 1440, 3)


In [3]:
vid_t, vid_list, fps = load_video('/home/assafsin/projects/vdm_optical_flow/self_warp/teaser_video/MonkeyJumpingOnTheBed_our_i2v_output_sdedit2_blend4_seed0.mp4')
print(fps, vid_t.shape, np.array(vid_list).shape)
write_video_imageio(np.array(vid_list), '/home/assafsin/projects/vdm_optical_flow/self_warp/teaser_video/MonkeyJumpingOnTheBed_our_i2v_output_sdedit2_blend4_seed0_fps=24.mp4', fps=24)

Min: 0.0, Max: 1.0
{'fps': 16.0, 'new_fps': 16.0} torch.Size([1, 3, 81, 512, 768]) (81, 512, 768, 3)


In [4]:
vid_t, vid_list, fps = load_video('/home/assafsin/projects/vdm_optical_flow/self_warp/teaser_video/Owl_our_i2v_output_sdedit2_blend4_seed0.mp4')
print(fps, vid_t.shape, np.array(vid_list).shape)
write_video_imageio(np.array(vid_list), '/home/assafsin/projects/vdm_optical_flow/self_warp/teaser_video/Owl_our_i2v_output_sdedit2_blend4_seed0_fps=24.mp4', fps=24)

Min: 0.0, Max: 1.0
{'fps': 16.0, 'new_fps': 16.0} torch.Size([1, 3, 81, 512, 768]) (81, 512, 768, 3)


In [5]:
vid_t, vid_list, fps = load_video('/home/assafsin/projects/vdm_optical_flow/self_warp/teaser_video/rhino_our_i2v_output_sdedit2_blend5_seed0.mp4')
print(fps, vid_t.shape, np.array(vid_list).shape)
write_video_imageio(np.array(vid_list), '/home/assafsin/projects/vdm_optical_flow/self_warp/teaser_video/rhino_our_i2v_output_sdedit2_blend5_seed0_fps=24.mp4', fps=24)

Min: 0.0, Max: 1.0
{'fps': 16.0, 'new_fps': 16.0} torch.Size([1, 3, 81, 512, 768]) (81, 512, 768, 3)


In [3]:
vid_t, vid_list, fps = load_video('/home/assafsin/projects/vdm_optical_flow/for_paper-amir/videos_for_appearance_control/concat_motion_left.mp4')
print(fps, vid_t.shape, np.array(vid_list).shape)
write_video_imageio(np.array(vid_list), '/home/assafsin/projects/vdm_optical_flow/for_paper-amir/videos_for_appearance_control/concat_motion_left_fps=24.mp4', fps=24)

Min: 0.0, Max: 1.0
{'fps': 16.0, 'new_fps': 16.0} torch.Size([1, 3, 81, 480, 1440]) (81, 480, 1440, 3)


In [4]:
import numpy as np
polygon = np.load('/home/assafsin/projects/vdm_optical_flow/self_warp/teaser_video/Owl_polygons.npy')

In [ ]:
len(polygon), len(polygon[0]), polygon[0].shape # (T, N, 2)

(1, 49, (7, 2))

In [6]:
polygon[0].shape

(49, 7, 2)

In [8]:
new_polygon.shape

(1, 49, 7, 2)

In [10]:
# reorder the points in the polygons, making the first point the last point, and shifting all other points one position to the left
new_polygon =  np.zeros_like(polygon[0])
new_polygon[:, :-1, :] = polygon[0][:, 1:, :]
new_polygon[:, -1, :] = polygon[0][:, 0, :]
new_polygon.shape
# save the new polygons
np.save('/home/assafsin/projects/vdm_optical_flow/self_warp/teaser_video/Owl_polygons_reordered.npy', [new_polygon])

In [ ]:
vid_t, vid_list, fps = load_video('/home/assafsin/projects/vdm_optical_flow/for_paper-amir/videos_for_appearance_control/concat_motion_left.mp4')
print(fps, vid_t.shape, np.array(vid_list).shape)
write_video_imageio(np.array(vid_list), '/home/assafsin/projects/vdm_optical_flow/for_paper-amir/videos_for_appearance_control/concat_motion_left_fps=24.mp4', fps=24)

Min: 0.0, Max: 1.0
{'fps': 16.0, 'new_fps': 16.0} torch.Size([1, 3, 81, 480, 1440]) (81, 480, 1440, 3)


##### teaser_video_camera_control

In [5]:
vid_t, vid_list, fps = load_video('/home/assafsin/projects/vdm_optical_flow/self_warp/teaser_video_camera_control/Bridge1_our_i2v_output_sdedit2_blend4_seed0.mp4')
print(fps, vid_t.shape, np.array(vid_list).shape)
write_video_imageio(np.array(vid_list), '/home/assafsin/projects/vdm_optical_flow/self_warp/teaser_video_camera_control/Bridge1_our_i2v_output_sdedit2_blend4_seed0_fps=24.mp4', fps=24)

Min: 0.0, Max: 1.0
{'fps': 16.0, 'new_fps': 16.0} torch.Size([1, 3, 81, 512, 768]) (81, 512, 768, 3)


In [8]:
vid_t, vid_list, fps = load_video('/home/assafsin/projects/vdm_optical_flow/self_warp/teaser_video_camera_control/Bridge1_warped_81.mp4')
print(fps, vid_t.shape, np.array(vid_list).shape)
write_video_imageio(np.array(vid_list), '/home/assafsin/projects/vdm_optical_flow/self_warp/teaser_video_camera_control/Bridge1_warped_81_fps=24.mp4', fps=24)

Min: 0.0, Max: 1.0
{'fps': 30.0, 'new_fps': 30.0} torch.Size([1, 3, 81, 480, 720]) (81, 480, 720, 3)


In [4]:
vid_t, vid_list, fps = load_video('/home/assafsin/projects/vdm_optical_flow/self_warp/teaser_video_camera_control/Bridge2_our_i2v_output_sdedit2_blend5_seed1.mp4')
print(fps, vid_t.shape, np.array(vid_list).shape)
write_video_imageio(np.array(vid_list), '/home/assafsin/projects/vdm_optical_flow/self_warp/teaser_video_camera_control/Bridge2_our_i2v_output_sdedit2_blend5_seed1_fps=24.mp4', fps=24)

Min: 0.0, Max: 1.0
{'fps': 16.0, 'new_fps': 16.0} torch.Size([1, 3, 81, 512, 768]) (81, 512, 768, 3)


In [9]:
vid_t, vid_list, fps = load_video('/home/assafsin/projects/vdm_optical_flow/self_warp/teaser_video_camera_control/Bridge2_warped.mp4')
print(fps, vid_t.shape, np.array(vid_list).shape)
write_video_imageio(np.array(vid_list), '/home/assafsin/projects/vdm_optical_flow/self_warp/teaser_video_camera_control/Bridge2_warped_fps=24.mp4', fps=24)

Min: 0.0, Max: 0.9803921580314636
{'fps': 30.0, 'new_fps': 30.0} torch.Size([1, 3, 81, 480, 720]) (81, 480, 720, 3)


In [6]:
vid_t, vid_list, fps = load_video('/home/assafsin/projects/vdm_optical_flow/self_warp/teaser_video_camera_control/concertstage_our_i2v_output_sdedit2_blend4_seed1.mp4')
print(fps, vid_t.shape, np.array(vid_list).shape)
write_video_imageio(np.array(vid_list), '/home/assafsin/projects/vdm_optical_flow/self_warp/teaser_video_camera_control/concertstage_our_i2v_output_sdedit2_blend4_seed1_fps=24.mp4', fps=24)

Min: 0.0, Max: 1.0
{'fps': 16.0, 'new_fps': 16.0} torch.Size([1, 3, 81, 512, 768]) (81, 512, 768, 3)


In [10]:
vid_t, vid_list, fps = load_video('/home/assafsin/projects/vdm_optical_flow/self_warp/teaser_video_camera_control/ConcertStage_warped_81.mp4')
print(fps, vid_t.shape, np.array(vid_list).shape)
write_video_imageio(np.array(vid_list), '/home/assafsin/projects/vdm_optical_flow/self_warp/teaser_video_camera_control/ConcertStage_warped_81_fps=24.mp4', fps=24)

Min: 0.0, Max: 1.0
{'fps': 30.0, 'new_fps': 30.0} torch.Size([1, 3, 81, 480, 720]) (81, 480, 720, 3)


In [7]:
vid_t, vid_list, fps = load_video('/home/assafsin/projects/vdm_optical_flow/self_warp/teaser_video_camera_control/RiverOcean_ours_wan.mp4')
print(fps, vid_t.shape, np.array(vid_list).shape)
write_video_imageio(np.array(vid_list), '/home/assafsin/projects/vdm_optical_flow/self_warp/teaser_video_camera_control/RiverOcean_ours_wan_fps=24.mp4', fps=24)

Min: 0.0, Max: 1.0
{'fps': 16.0, 'new_fps': 16.0} torch.Size([1, 3, 81, 512, 768]) (81, 512, 768, 3)


In [7]:
vid_t, vid_list, fps = load_video('/home/assafsin/projects/vdm_optical_flow/self_warp/teaser_video_camera_control/RiverOcean_warped_81.mp4')
print(fps, vid_t.shape, np.array(vid_list).shape)
write_video_imageio(np.array(vid_list), '/home/assafsin/projects/vdm_optical_flow/self_warp/teaser_video_camera_control/RiverOcean_warped_81_fps=24.mp4', fps=24)

Min: 0.0, Max: 1.0
{'fps': 30.0, 'new_fps': 30.0} torch.Size([1, 3, 81, 480, 720]) (81, 480, 720, 3)


In [2]:
# create a function that load an image and a depth map image (already as png) of the same shape (H, W, C), and returns a video numpy array of shape (81, H, W, C), where the video is the follows:
# 27 * the original image + 27 * the depth map image + 27 * the original image.

import imageio
import numpy as np
import os

def create_video_from_images(original_image_path, depth_map_image_path):
    original_image = imageio.imread(original_image_path)
    depth_map_image = imageio.imread(depth_map_image_path)

    assert original_image.shape == depth_map_image.shape, "Original image and depth map must have the same shape"

    video = np.zeros((48, *original_image.shape), dtype=original_image.dtype)
    video[:18] = original_image
    video[18:30] = depth_map_image
    video[30:] = original_image

    return video

In [6]:
vid_depth = create_video_from_images("/home/assafsin/projects/vdm_optical_flow/self_warp/teaser_video_camera_control/images_depth_maps/Bridge1-image_720.png",
"/home/assafsin/projects/vdm_optical_flow/self_warp/teaser_video_camera_control/images_depth_maps/Bridge1-depth_map_log_scale.png")
print(vid_depth.shape)
write_video_imageio(vid_depth, '/home/assafsin/projects/vdm_optical_flow/self_warp/teaser_video_camera_control/Bridge1_depth_48_fps=24.mp4', fps=24)

(48, 480, 720, 3)


In [7]:
vid_depth = create_video_from_images("/home/assafsin/projects/vdm_optical_flow/self_warp/teaser_video_camera_control/images_depth_maps/Bridge2-image_720.png",
"/home/assafsin/projects/vdm_optical_flow/self_warp/teaser_video_camera_control/images_depth_maps/Bridge2-depth_map_log_scale.png")
print(vid_depth.shape)
write_video_imageio(vid_depth, '/home/assafsin/projects/vdm_optical_flow/self_warp/teaser_video_camera_control/Bridge2_depth_48_fps=24.mp4', fps=24)

(48, 480, 720, 3)


In [8]:
vid_depth = create_video_from_images("/home/assafsin/projects/vdm_optical_flow/self_warp/teaser_video_camera_control/images_depth_maps/ConcertStage-image_720.png",
"/home/assafsin/projects/vdm_optical_flow/self_warp/teaser_video_camera_control/images_depth_maps/ConcertStage-depth_map_log_scale.png")
print(vid_depth.shape)
write_video_imageio(vid_depth, '/home/assafsin/projects/vdm_optical_flow/self_warp/teaser_video_camera_control/ConcertStage_depth_48_fps=24.mp4', fps=24)

(48, 480, 720, 3)


In [7]:
import numpy as np


def _to_float01(x: np.ndarray) -> tuple[np.ndarray, dict]:
    info = {"orig_dtype": x.dtype}
    x = x.astype(np.float32)
    if np.issubdtype(info["orig_dtype"], np.integer):
        maxv = np.iinfo(info["orig_dtype"]).max
        x /= maxv
        info["scale"] = maxv
    else:
        info["scale"] = 1.0
    return x, info

def _from_float01(xf: np.ndarray, info: dict) -> np.ndarray:
    xf = np.clip(xf, 0.0, 1.0)
    out = (xf * info["scale"]).round()
    return out.astype(info["orig_dtype"])

def make_depth_fade_video(
    image, 
    depth_map_rgb, 
    frames_total: int = 48, 
    fade_in_frames: int = 24
) -> np.ndarray:
    """
    Create a (frames_total, H, W, C) video where the RGB depth map is
    overlaid on the image with a fade-in then fade-out.

    Parameters
    ----------
    image : str | np.ndarray
        Path to PNG or array of shape (H, W, C).
    depth_map_rgb : str | np.ndarray
        Path to PNG or array of shape (H, W, C) in a yellow/green RGB colormap.
    frames_total : int
        Total number of frames (default 48).
    fade_in_frames : int
        Frames used for fade-in (rest are fade-out).

    Returns
    -------
    video : np.ndarray
        (frames_total, H, W, C) with same dtype as the input image.
    """
    img = imageio.imread(image)
    dep = imageio.imread(depth_map_rgb)

    if img.ndim != 3 or dep.ndim != 3:
        raise ValueError("image and depth_map_rgb must have shape (H, W, C)")
    if img.shape != dep.shape:
        raise ValueError(f"Shapes must match, got {img.shape} vs {dep.shape}")
    if not (0 < fade_in_frames < frames_total):
        raise ValueError("fade_in_frames must be in [1, frames_total-1]")

    H, W, C = img.shape
    img_f, img_info = _to_float01(img)
    dep_f, _ = _to_float01(dep)  # use the colorized depth as the overlay

    # Alpha schedule: linear fade-in then fade-out
    fade_out_frames = frames_total - fade_in_frames
    alphas_in  = np.linspace(0.0, 1.0, fade_in_frames, endpoint=False)  # 0 -> just before 1
    alphas_out = np.linspace(1.0, 0.0, fade_out_frames, endpoint=True)  # 1 -> 0 inclusive
    alphas = np.concatenate([alphas_in, alphas_out]).astype(np.float32)

    # Blend per frame: (1-a)*image + a*depth_overlay (depth overlay is RGB)
    video = np.empty((frames_total, H, W, C), dtype=np.float32)
    for t, a in enumerate(alphas):
        video[t] = (1.0 - a) * img_f + a * dep_f

    return _from_float01(video, img_info)


In [9]:
video_depth_fade = make_depth_fade_video("/home/assafsin/projects/vdm_optical_flow/self_warp/teaser_video_camera_control/images_depth_maps/Bridge1-image_720.png",
"/home/assafsin/projects/vdm_optical_flow/self_warp/teaser_video_camera_control/images_depth_maps/Bridge1-depth_map_log_scale.png")
print(video_depth_fade.shape)
write_video_imageio(video_depth_fade, '/home/assafsin/projects/vdm_optical_flow/self_warp/teaser_video_camera_control/Bridge1_depth_fade_48_fps=24.mp4', fps=24)

(48, 480, 720, 3)


In [10]:
vid_depth = make_depth_fade_video("/home/assafsin/projects/vdm_optical_flow/self_warp/teaser_video_camera_control/images_depth_maps/Bridge2-image_720.png",
"/home/assafsin/projects/vdm_optical_flow/self_warp/teaser_video_camera_control/images_depth_maps/Bridge2-depth_map_log_scale.png")
print(vid_depth.shape)
write_video_imageio(vid_depth, '/home/assafsin/projects/vdm_optical_flow/self_warp/teaser_video_camera_control/Bridge2_depth_fade_48_fps=24.mp4', fps=24)

(48, 480, 720, 3)


In [11]:
vid_depth = make_depth_fade_video("/home/assafsin/projects/vdm_optical_flow/self_warp/teaser_video_camera_control/images_depth_maps/ConcertStage-image_720.png",
"/home/assafsin/projects/vdm_optical_flow/self_warp/teaser_video_camera_control/images_depth_maps/ConcertStage-depth_map_log_scale.png")
print(vid_depth.shape)
write_video_imageio(vid_depth, '/home/assafsin/projects/vdm_optical_flow/self_warp/teaser_video_camera_control/ConcertStage_depth_fade_48_fps=24.mp4', fps=24)

(48, 480, 720, 3)


In [ ]:
vid_t, vid_list, fps = load_video('/home/assafsin/projects/vdm_optical_flow/for_paper-amir/videos_for_appearance_control/ChameleonColor_ours.mp4')
print(fps, vid_t.shape, np.array(vid_list).shape)
write_video_imageio(np.array(vid_list), '/home/assafsin/projects/vdm_optical_flow/self_warp/teaser_video_appearance_control/ChameleonColor_ours_wan_fps=24.mp4', fps=24)

Min: 0.0, Max: 1.0
{'fps': 16.0, 'new_fps': 16.0} torch.Size([1, 3, 81, 512, 768]) (81, 512, 768, 3)


In [2]:
vid_t, vid_list, fps = load_video('/home/assafsin/projects/vdm_optical_flow/self_warp/teaser_video_appearance_control/car_crush/par_5_10_seed1.mp4')
print(fps, vid_t.shape, np.array(vid_list).shape)
write_video_imageio(np.array(vid_list), '/home/assafsin/projects/vdm_optical_flow/self_warp/teaser_video_appearance_control/car_crush/par_5_10_seed1_fps=24.mp4', fps=24)

Min: 0.0, Max: 1.0
{'fps': 16.0, 'new_fps': 16.0} torch.Size([1, 3, 81, 512, 768]) (81, 512, 768, 3)


In [4]:
vid_t, vid_list, fps = load_video('/home/assafsin/projects/vdm_optical_flow/self_warp/teaser_video/teaser_video_5_v1_trimmed.mp4')
print(fps, vid_t.shape, np.array(vid_list).shape)
write_video_imageio(np.array(vid_list), '/home/assafsin/projects/vdm_optical_flow/self_warp/teaser_video/teaser_video_5_v1_trimmed_fps=16.mp4', fps=16)

Min: 0.0, Max: 1.0
{'fps': 21.428571428571427, 'new_fps': 21.428571428571427} torch.Size([1, 3, 85, 1080, 1920]) (85, 1080, 1920, 3)


IMAGEIO FFMPEG_WRITER WARNING: input image is not divisible by macro_block_size=16, resizing from (1920, 1080) to (1920, 1088) to ensure video compatibility with most codecs and players. To prevent resizing, make your input image divisible by the macro_block_size or set the macro_block_size to 1 (risking incompatibility).


In [2]:
# iterate over videos in /home/assafsin/projects/vdm_optical_flow/self_warp/twitter_post/, convert them to fps=16, and save them with the suffix _fps=16.mp4
import os
data_dir = "/home/assafsin/projects/vdm_optical_flow/self_warp/twitter_post/"
file_list = os.listdir(data_dir)
for filename in file_list:
    if filename.endswith(".mp4"):
        vid_t, vid_list, fps = load_video(os.path.join(data_dir, filename))
        print(f"Converting {filename} to fps=16")
        print(fps, vid_t.shape, np.array(vid_list).shape)
        new_name = filename.replace("_fps=24.mp4", ".mp4").replace(".mp4", "_fps=16.mp4") # to avoid double _fps=*
        write_video_imageio(np.array(vid_list), os.path.join(data_dir, new_name), fps=16)

Min: 0.0, Max: 1.0
Converting Bridge1_our_i2v_output_sdedit2_blend4_seed0.mp4 to fps=16
{'fps': 16.0, 'new_fps': 16.0} torch.Size([1, 3, 81, 512, 768]) (81, 512, 768, 3)
Min: 0.0, Max: 1.0
Converting Bridge1_warped_81_fps=24.mp4 to fps=16
{'fps': 24.0, 'new_fps': 24.0} torch.Size([1, 3, 81, 480, 720]) (81, 480, 720, 3)
Min: 0.0, Max: 1.0
Converting CarCrush_overlayed_edges_mac_cursor_star_magic_tool_ew=5_s=morph_sr=2_ex=6_interp49to81_cursor_customtip_cursize=40.mp4 to fps=16
{'fps': 16.0, 'new_fps': 16.0} torch.Size([1, 3, 81, 480, 720]) (81, 480, 720, 3)
Min: 0.0, Max: 1.0
Converting CarCrush_par_5_10_seed1_fps=24.mp4 to fps=16
{'fps': 24.0, 'new_fps': 24.0} torch.Size([1, 3, 81, 512, 768]) (81, 512, 768, 3)
Min: 0.0, Max: 1.0
Converting ChameleonColor_ours_wan_fps=24.mp4 to fps=16
{'fps': 24.0, 'new_fps': 24.0} torch.Size([1, 3, 81, 512, 768]) (81, 512, 768, 3)
Min: 0.0, Max: 1.0
Converting ChameleonColor_overlayed_edges_color_bucket_purple_ew=5_s=morph_sr=2_ex=6_interp49to81_cursor

In [5]:
vid_t, vid_list, fps = load_video('/home/assafsin/projects/vdm_optical_flow/self_warp/twitter_post/sand-motion_prompting_recolored.mp4')
print(fps, vid_t.shape, np.array(vid_list).shape)
write_video_imageio(np.array(vid_list), '/home/assafsin/projects/vdm_optical_flow/self_warp/twitter_post/sand-motion_prompting_recolored.mp4', fps=fps['fps'])


Min: 0.0, Max: 1.0
{'fps': 16.0, 'new_fps': 16.0} torch.Size([1, 3, 77, 480, 128]) (77, 480, 128, 3)


# split car-crash mask to two masks

# delete unnecessary files

In [2]:
# To recursively iterate over `/home/assafsin/projects/vdm_optical_flow/experiments-t2v` and delete all subdirectories containing 'wg=0' in their name, you can use the following Python code:
import os
import shutil

root_dir = '/home/assafsin/projects/vdm_optical_flow/experiments-t2v'

for dirpath, dirnames, filenames in os.walk(root_dir):
    for dirname in dirnames:
        if 'wg=0' in dirname:
            full_path = os.path.join(dirpath, dirname)
            print(f"Deleting: {full_path}")
            shutil.rmtree(full_path)

Deleting: /home/assafsin/projects/vdm_optical_flow/experiments-t2v/locomotive_speed_t2v-flow_guid/mg=0_gc=0_steps=200_rec_steps=5_wg=0_seed=42
Deleting: /home/assafsin/projects/vdm_optical_flow/experiments-t2v/locomotive_speed_t2v-flow_guid/mg=0_gc=0_steps=100_rec_steps=1_wg=0_seed=42
Deleting: /home/assafsin/projects/vdm_optical_flow/experiments-t2v/locomotive_speed_t2v-flow_guid/mg=0_gc=0_steps=200_rec_steps=1_wg=0_seed=42


Deleting: /home/assafsin/projects/vdm_optical_flow/experiments-t2v/locomotive_speed_t2v-flow_guid/mg=1000_gc=2000_steps=200_rec_steps=5_wg=0_seed=42
Deleting: /home/assafsin/projects/vdm_optical_flow/experiments-t2v/locomotive_speed_t2v-flow_guid/mg=5000_gc=2000_steps=200_rec_steps=5_wg=0_seed=42
Deleting: /home/assafsin/projects/vdm_optical_flow/experiments-t2v/locomotive_speed_t2v-flow_guid/mg=0_gc=0_steps=100_rec_steps=1_wg=0_seed=123
Deleting: /home/assafsin/projects/vdm_optical_flow/experiments-t2v/locomotive_speed_t2v-flow_guid/mg=0_gc=0_steps=100_rec_steps=1_wg=0_seed=456
Deleting: /home/assafsin/projects/vdm_optical_flow/experiments-t2v/locomotive_speed_t2v-flow_guid/mg=0_gc=0_steps=200_rec_steps=1_wg=0_seed=456
Deleting: /home/assafsin/projects/vdm_optical_flow/experiments-t2v/locomotive_speed_t2v-flow_guid/mg=0_gc=0_steps=200_rec_steps=1_wg=0_seed=123
Deleting: /home/assafsin/projects/vdm_optical_flow/experiments-t2v/car_turn_t2v-flow_guid/mg=0_gc=0_steps=200_rec_steps=1_wg=0

In [11]:
vid_t, vid_list, fps = load_video("//gipdeep_home/ganz_04/noam/video_flow/motionwarp/MotionPro/data/MC-Amir/samples/BloodMoon_video.mp4")
print(fps, vid_t.shape, np.array(vid_list).shape)

Min: 0.05098039284348488, Max: 0.9725490212440491
{'fps': 30.30165305286752, 'new_fps': 30.30165305286752} torch.Size([1, 3, 229, 1080, 1624]) (229, 1080, 1624, 3)


In [ ]:
vid_t, vid_list, fps = load_video('/home/assafsin/projects/vdm_optical_flow/amir_samples/BloodMoon_video.mp4')
print(fps, vid_t.shape, np.array(vid_list).shape)

Min: 0.003921568859368563, Max: 0.9764705896377563
{'fps': 24.0, 'new_fps': 24.0} torch.Size([1, 3, 229, 1088, 1632]) (229, 1088, 1632, 3)


In [3]:
vid_t, vid_list, fps = load_video('/home/assafsin/projects/vdm_optical_flow/amir_samples/ICLRClouds_video.mp4')
print(fps, vid_t.shape, np.array(vid_list).shape)

Min: 0.11372549086809158, Max: 1.0
{'fps': 24.0, 'new_fps': 24.0} torch.Size([1, 3, 229, 1088, 1632]) (229, 1088, 1632, 3)


In [4]:
vid_t, vid_list, fps = load_video('/home/assafsin/projects/vdm_optical_flow/amir_samples_camera_control/ConcertCrowd_warped_video.mp4')
print(fps, vid_t.shape, np.array(vid_list).shape)

Min: 0.0, Max: 1.0
{'fps': 16.0, 'new_fps': 16.0} torch.Size([1, 3, 49, 480, 720]) (49, 480, 720, 3)


In [2]:
for video_name in ["chameleon_sf=0_f=49", "chameleon_2_sf=0_f=49"]:
    vid_t, vid_list, fps = load_video(f'/home/assafsin/projects/vdm_optical_flow/data/{video_name}.mp4',
                                    res_h=480, res_w=720)
    # write the video to a file
    write_video_imageio(np.array(vid_list), f'/home/assafsin/projects/vdm_optical_flow/data/{video_name}_video.mp4', fps=8)
    # save the first frame as an image
from PIL import Image
for img_name in ["chameleon", "chameleon_2"]:
    img = Image.open(f'/home/assafsin/projects/vdm_optical_flow/data/{img_name}_video_frame_0.png')
    img = img.resize((720, 480))
    img.save(f'/home/assafsin/projects/vdm_optical_flow/data/{img_name}_video_frame_0_resized.png')

Min: 0.0, Max: 1.0
Min: 0.0, Max: 1.0


In [1]:
msk_npy = "/data/DL3DV-10k/10K/992fd7cf41ecb7969b51f68fa137142ed17051a2fcfb91ec0f761aa5a4bb0faa/images_4_masks_frame_00001_frame_00049.npy"
warp_npy = "/data/DL3DV-10k/10K/992fd7cf41ecb7969b51f68fa137142ed17051a2fcfb91ec0f761aa5a4bb0faa/images_4_warped_frame_00001_frame_00049.npy"

In [3]:
import numpy as np
# load numPy arrays
msk = np.load(msk_npy)
warp = np.load(warp_npy)
msk.shape, warp.shape, np.unique(msk), np.min(warp), np.max(warp)

((49, 540, 960, 3), (49, 540, 960, 3), array([  0, 255], dtype=uint8), 0, 253)

In [8]:
write_video_imageio(msk, 'edge_masks/images_4_masks_frame_00001_frame_00049.mp4', fps=8)
write_video_imageio(warp, 'edge_masks/images_4_warped_frame_00001_frame_00049.mp4', fps=8)

IMAGEIO FFMPEG_WRITER WARNING: input image is not divisible by macro_block_size=16, resizing from (960, 540) to (960, 544) to ensure video compatibility with most codecs and players. To prevent resizing, make your input image divisible by the macro_block_size or set the macro_block_size to 1 (risking incompatibility).


IMAGEIO FFMPEG_WRITER WARNING: input image is not divisible by macro_block_size=16, resizing from (960, 540) to (960, 544) to ensure video compatibility with most codecs and players. To prevent resizing, make your input image divisible by the macro_block_size or set the macro_block_size to 1 (risking incompatibility).


In [12]:
import cv2
import os

def save_first_frame(mp4_path, frame_path):
    cap = cv2.VideoCapture(mp4_path)
    ret, frame = cap.read()
    if ret:
        cv2.imwrite(frame_path, frame)
    cap.release()

def save_last_frame(mp4_path, frame_path):
    cap = cv2.VideoCapture(mp4_path)
    last_frame = None
    while True:
        ret, frame = cap.read()
        if not ret:
            break
        last_frame = frame

    if last_frame is not None:
        cv2.imwrite(frame_path, last_frame)
    cap.release()

In [ ]:
mp4_path = "/home/assafsin/projects/vdm_optical_flow/self_warp/edge_masks/chameleon_mask_dy=28.mp4"
frame_path = "/home/assafsin/projects/vdm_optical_flow/self_warp/edge_masks/chameleon_mask_dy=28_frame_0.png"

save_first_frame(mp4_path, frame_path)

In [ ]:
mp4_path = "/home/assafsin/projects/vdm_optical_flow/data/chameleon_2_v2_mask.mp4"
frame_path = "/home/assafsin/projects/vdm_optical_flow/self_warp/edge_masks/chameleon_2_v2_mask_frame_0.png"

save_first_frame(mp4_path, frame_path)

In [ ]:
mp4_path = "/home/assafsin/projects/vdm_optical_flow/data/chameleon_2_v2_mask.mp4"
frame_path = "/home/assafsin/projects/vdm_optical_flow/self_warp/edge_masks/chameleon_2_v2_mask_frame_last.png"

save_last_frame(mp4_path, frame_path)

In [7]:
mp4_path = "/home/assafsin/projects/vdm_optical_flow/data/chameleon_sf=0_f=49.mp4"
frame_path = "/home/assafsin/projects/vdm_optical_flow/self_warp/edge_masks/chameleon_frame_last.png"

save_last_frame(mp4_path, frame_path)

In [6]:
mp4_path = "/home/assafsin/projects/vdm_optical_flow/data/chameleon_2_v2_sf=0_f=49.mp4"
frame_path = "/home/assafsin/projects/vdm_optical_flow/self_warp/edge_masks/chameleon_2_v2_sf=0_f=49_frame_last.png"

save_last_frame(mp4_path, frame_path)

# Copy MC-Bench masks to rebuttal folder

In [1]:
videos_list = ["ZZW_2024_07_08_1629_20_02", 
"ZZW_2024_07_08_1647_20_01", 
"ZZW_2024_07_08_1653_20_02", 
"ZZW_2024_07_08_1654_20_01", 
"ZZW_2024_07_08_1655_20_01", 
"ZZW_2024_07_08_1705_20_01", 
"ZZW_2024_07_08_1708_20_01", 
"ZZW_2024_07_08_1711_20_02", 
"ZZW_2024_07_09_0126_20_05", 
"ZZW_2024_07_09_0129_20_03", 
"ZZW_2024_07_09_0130_20_01", 
"ZZW_2024_07_09_0134_20_01", 
"ZZW_2024_07_09_0137_20_01", ]

In [7]:
vid_t, vid_np, fps = load_video_simple('/home/assafsin/projects/vdm_optical_flow/for_paper-amir/rebuttal/MC-Bench/ZZW_2024_07_08_1629_20_02/concatenated_fixed.mp4')
print(f"Video shape: {vid_t.shape}, numpy shape: {vid_np.shape}, FPS: {fps}")

Min: 0.0, Max: 1.0
Video shape: torch.Size([1, 3, 14, 320, 2560]), numpy shape: (14, 320, 2560, 3), FPS: 7.0


In [6]:
import shutil

source_dir = "/gipdeep_home/ganz_04/noam/video_flow/motionwarp/MotionPro/data/MC-Bench_control-only_single-traj/object_control/"
target_dir = "/home/assafsin/projects/vdm_optical_flow/for_paper-amir/rebuttal/MC-Bench"
for video_name in videos_list:
    mask_mp4 = f"{source_dir}{video_name}//mask_video_framec-25-multipoints.mp4"
    target_mp4 = f"{target_dir}/{video_name}/mask_video_framec-25-multipoints.mp4"
    # copy the file
    print(f"Copying {mask_mp4} to {target_mp4}")
    shutil.copyfile(mask_mp4, target_mp4)


Copying /gipdeep_home/ganz_04/noam/video_flow/motionwarp/MotionPro/data/MC-Bench_control-only_single-traj/object_control/ZZW_2024_07_08_1629_20_02//mask_video_framec-25-multipoints.mp4 to /home/assafsin/projects/vdm_optical_flow/for_paper-amir/rebuttal/MC-Bench/ZZW_2024_07_08_1629_20_02/mask_video_framec-25-multipoints.mp4
Copying /gipdeep_home/ganz_04/noam/video_flow/motionwarp/MotionPro/data/MC-Bench_control-only_single-traj/object_control/ZZW_2024_07_08_1647_20_01//mask_video_framec-25-multipoints.mp4 to /home/assafsin/projects/vdm_optical_flow/for_paper-amir/rebuttal/MC-Bench/ZZW_2024_07_08_1647_20_01/mask_video_framec-25-multipoints.mp4
Copying /gipdeep_home/ganz_04/noam/video_flow/motionwarp/MotionPro/data/MC-Bench_control-only_single-traj/object_control/ZZW_2024_07_08_1653_20_02//mask_video_framec-25-multipoints.mp4 to /home/assafsin/projects/vdm_optical_flow/for_paper-amir/rebuttal/MC-Bench/ZZW_2024_07_08_1653_20_02/mask_video_framec-25-multipoints.mp4
Copying /gipdeep_home/gan

In [2]:
vid_t, vid_np, fps = load_video_simple('/home/assafsin/projects/vdm_optical_flow/presentation/TimeSquaresV5_video.mp4')
print(f"Video shape: {vid_t.shape}, numpy shape: {vid_np.shape}, FPS: {fps}")

Min: 0.0, Max: 1.0
Video shape: torch.Size([1, 3, 81, 480, 720]), numpy shape: (81, 480, 720, 3), FPS: 24.0


In [3]:
vid_t, vid_np, fps = load_video_simple('/home/assafsin/projects/vdm_optical_flow/for_paper-amir/videos_for_appearance_control/TimeToMoveVideo/Presentation1_mask.mp4')
print(f"Video shape: {vid_t.shape}, numpy shape: {vid_np.shape}, FPS: {fps}")

Min: 0.0, Max: 1.0
Video shape: torch.Size([1, 3, 81, 480, 720]), numpy shape: (81, 480, 720, 3), FPS: 24.0


In [3]:
vid_t, vid_np, fps = load_video_simple('/home/assafsin/projects/vdm_optical_flow/presentation/TimeSquaresV5_video.mp4')
print(f"Video shape: {vid_t.shape}, numpy shape: {vid_np.shape}, FPS: {fps}")
write_video_imageio(vid_np, '/home/assafsin/projects/vdm_optical_flow/presentation/TimeSquaresV5_video_fps=16.mp4', fps=16)

Min: 0.0, Max: 1.0
Video shape: torch.Size([1, 3, 81, 480, 720]), numpy shape: (81, 480, 720, 3), FPS: 24.0


In [3]:
vid_t, vid_np, fps = load_video_simple('/home/assafsin/projects/TTM/examples/cutdrag_wan_Cocktail/motion_signal.mp4')
print(f"Warped - Video shape: {vid_t.shape}, numpy shape: {vid_np.shape}, FPS: {fps}")
vid_t, vid_np, fps = load_video_simple('/home/assafsin/projects/TTM/examples/cutdrag_wan_Birds/motion_signal.mp4')
print(f"Warped - Video shape: {vid_t.shape}, numpy shape: {vid_np.shape}, FPS: {fps}")
vid_t, vid_np, fps = load_video_simple('/home/assafsin/projects/TTM/examples/cutdrag_wan_Gardening/motion_signal.mp4')
print(f"Warped - Video shape: {vid_t.shape}, numpy shape: {vid_np.shape}, FPS: {fps}")
vid_t, vid_np, fps = load_video_simple('/home/assafsin/projects/TTM/examples/cutdrag_wan_Hamburger/motion_signal.mp4')
print(f"Warped - Video shape: {vid_t.shape}, numpy shape: {vid_np.shape}, FPS: {fps}")
vid_t, vid_np, fps = load_video_simple('/home/assafsin/projects/TTM/examples/cutdrag_wan_Monkey/motion_signal.mp4')
print(f"Warped - Video shape: {vid_t.shape}, numpy shape: {vid_np.shape}, FPS: {fps}")

Min: 0.0, Max: 1.0
Warped - Video shape: torch.Size([1, 3, 81, 480, 720]), numpy shape: (81, 480, 720, 3), FPS: 24.0
Min: 0.0, Max: 1.0
Warped - Video shape: torch.Size([1, 3, 81, 480, 720]), numpy shape: (81, 480, 720, 3), FPS: 24.0
Min: 0.0, Max: 1.0
Warped - Video shape: torch.Size([1, 3, 81, 480, 720]), numpy shape: (81, 480, 720, 3), FPS: 24.0
Min: 0.0, Max: 1.0
Warped - Video shape: torch.Size([1, 3, 81, 480, 720]), numpy shape: (81, 480, 720, 3), FPS: 24.0
Min: 0.0, Max: 1.0
Warped - Video shape: torch.Size([1, 3, 81, 480, 720]), numpy shape: (81, 480, 720, 3), FPS: 16.0


In [5]:
vid_t, vid_np, fps = load_video_simple('/home/assafsin/projects/TTM/examples/nvidia_car_demo/motion_signal.mp4')
print(f"Warped - Video shape: {vid_t.shape}, numpy shape: {vid_np.shape}, FPS: {fps}")
vid_t, vid_np, fps = load_video_simple('/home/assafsin/projects/TTM/examples/cutdrag_cog_Monkey/motion_signal.mp4')
print(f"Ours cog - Video shape: {vid_t.shape}, numpy shape: {vid_np.shape}, FPS: {fps}")
vid_t, vid_np, fps = load_video_simple('/home/assafsin/projects/TTM/outputs/cog_monkey.mp4')
print(f"Ours cog - Video shape: {vid_t.shape}, numpy shape: {vid_np.shape}, FPS: {fps}")

Min: 0.0, Max: 1.0
Warped - Video shape: torch.Size([1, 3, 161, 720, 1280]), numpy shape: (161, 720, 1280, 3), FPS: 30.0
Min: 0.0, Max: 0.9921568632125854
Ours cog - Video shape: torch.Size([1, 3, 49, 480, 720]), numpy shape: (49, 480, 720, 3), FPS: 8.0
Min: 0.0, Max: 0.9882352948188782
Ours cog - Video shape: torch.Size([1, 3, 49, 480, 720]), numpy shape: (49, 480, 720, 3), FPS: 8.0


In [7]:
vid_t, vid_list, fps = load_video('/home/assafsin/projects/TTM/examples/nvidia_car_demo/motion_signal.mp4',
                                res_h=480, res_w=720, start_frame=0, max_frames=81)
write_video_imageio(np.array(vid_list), '/home/assafsin/projects/TTM/examples/nvidia_car_demo/motion_signal_sf=0_f=81_h_480_w=720.mp4', fps=8)

Min: 0.0, Max: 0.9725490212440491


In [10]:
vid_t, vid_list, fps = load_video('/home/assafsin/projects/TTM/examples/nvidia_car_demo/motion_signal.mp4',
                                res_h=480, res_w=720, start_frame=0, max_frames=81, fps_jump=2)
write_video_imageio(np.array(vid_list), '/home/assafsin/projects/TTM/examples/nvidia_car_demo/motion_signal_sf=0_f=81_fps_jump=2_h_480_w=720.mp4', fps=fps['new_fps'])

Min: 0.0, Max: 0.9764705896377563


In [14]:
mp4_path = "/home/assafsin/projects/TTM/examples/nvidia_car_demo/motion_signal_sf=0_f=81_fps_jump=2_h_480_w=720.mp4"
frame_path = "/home/assafsin/projects/TTM/examples/nvidia_car_demo/first_frame.png"

save_first_frame(mp4_path, frame_path)

In [9]:
fps

{'fps': 30.0, 'new_fps': 15.0}

In [11]:
vid_t, vid_np, fps = load_video_simple('/home/assafsin/projects/TTM/examples/nvidia_car_demo/motion_signal.mp4')
print(f"Warped - Video shape: {vid_t.shape}, numpy shape: {vid_np.shape}, FPS: {fps}")
vid_t, vid_np, fps = load_video_simple('/home/assafsin/projects/TTM/examples/nvidia_car_demo/motion_signal_sf=0_f=81_fps_jump=2_h_480_w=720.mp4')
print(f"Warped - Video shape: {vid_t.shape}, numpy shape: {vid_np.shape}, FPS: {fps}")
vid_t, vid_np, fps = load_video_simple('/home/assafsin/projects/TTM/examples/cutdrag_wan_Cocktail/motion_signal.mp4')
print(f"Warped - Video shape: {vid_t.shape}, numpy shape: {vid_np.shape}, FPS: {fps}")

Min: 0.0, Max: 1.0
Warped - Video shape: torch.Size([1, 3, 161, 720, 1280]), numpy shape: (161, 720, 1280, 3), FPS: 30.0
Min: 0.0, Max: 0.9803921580314636
Warped - Video shape: torch.Size([1, 3, 81, 480, 720]), numpy shape: (81, 480, 720, 3), FPS: 15.0
Min: 0.0, Max: 1.0
Warped - Video shape: torch.Size([1, 3, 81, 480, 720]), numpy shape: (81, 480, 720, 3), FPS: 24.0
